In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import tensorflow as tf
import matplotlib.pyplot as plt
from architectures import ESPCN
from image_processing import downscale, load_and_preprocess, load_image_paths

# -----------------------------
# Configuración
# -----------------------------
UP_RATIO = 4
BATCH_SIZE = 16
EPOCHS = 100          # puedes subirlo cuando tengas GPU
HR_SIZE = (256, 256)  # tamaño de imagen HR para entrenamiento
DATA_FOLDER = "../../data/DIV2K_train_HR"  # ruta a tu dataset de imágenes HR
VALID_FOLDER = "../../data/DIV2K_valid_HR"  # ruta a tu dataset de imágenes HR

In [ ]:
print("GPUs visibles:", tf.config.list_physical_devices("GPU"))

In [ ]:
# -----------------------------
# Generador de datos
# -----------------------------
def build_dataset(image_paths, hr_size, up_ratio, batch_size, training=True):
    ds = tf.data.Dataset.from_tensor_slices(image_paths)

    if training:
        ds = ds.shuffle(buffer_size=len(image_paths))

    ds = ds.map(
        lambda p: load_and_preprocess(p, hr_size, up_ratio),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

# -----------------------------
# Mostrar imagenes predecidas por el modelo
# -----------------------------

def show_example(model, dataset, image_path=None, hr_size=(256, 256), up_ratio=4):
    """
    Muestra un ejemplo de la imagen desde la ruta de archivo(si se proporciona, sino se escoge de maenra aleatoria): LR → SR → HR
    """
    # Si `image_path` se proporciona, cargar esa imagen
    if image_path is not None:
        lr, hr = load_and_preprocess(image_path, hr_size, up_ratio)
        # Añadir dimensión de batch para que sea (1, h, w, 3)
        lr = tf.expand_dims(lr, axis=0)
        
    else:
        # Si no se proporciona, tomar una imagen aleatoria del dataset
        for lr_batch, hr_batch in dataset.take(1):
            lr = lr_batch[0:1]   # Seleccionar la primera imagen del lote
            hr = hr_batch[0]     # Seleccionar la primera imagen del lote
            

    # Predicción
    sr = model(lr, training=False)[0]

    # Convertir a numpy para visualización
    lr_np = lr[0].numpy()
    sr_np = sr.numpy()
    hr_np = hr.numpy()

    # Mostrar imágenes
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.title("Low Res (Input)")
    plt.imshow(lr_np)
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.title("Super Res (Prediction)")
    plt.imshow(sr_np)
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.title("High Res (Target)")
    plt.imshow(hr_np)
    plt.axis("off")

    plt.show()

def psnr(y_true, y_pred):
    return tf.image.psnr(y_true, y_pred, max_val=1.0)

def ssim(y_true, y_pred):
    return tf.image.ssim(y_true, y_pred, max_val=1.0)

class ShowExampleCallback(tf.keras.callbacks.Callback):

    def __init__(
        self,
        dataset,
        every_n_epochs=1,
        image_path=None,
        hr_size=(256, 256),
        up_ratio=4,
        num_examples=1
    ):
        super().__init__()
        self.dataset = dataset
        self.every_n_epochs = every_n_epochs
        self.image_path = image_path
        self.hr_size = hr_size
        self.up_ratio = up_ratio
        self.num_examples = num_examples

    def on_epoch_end(self, epoch, logs=None):
        current_epoch = epoch + 1

        if current_epoch % self.every_n_epochs != 0:
            return

        print(f"\nExample in epoch {current_epoch}\n")

        for _ in range(self.num_examples):
            show_example(
                model=self.model,
                dataset=self.dataset,
                image_path=self.image_path,
                hr_size=self.hr_size,
                up_ratio=self.up_ratio
            )

In [ ]:
# -----------------------------
# Crear y compilar modelo
# -----------------------------
model = ESPCN(up_ratio=UP_RATIO)
model.build((None, None, None, 3))
model.compile(optimizer="adam", loss="mae", metrics=[psnr, ssim], jit_compile=False)

model.summary()

# -----------------------------
# Entrenamiento
# -----------------------------

train_paths = load_image_paths(DATA_FOLDER)
val_paths   = load_image_paths(VALID_FOLDER)

train_ds = build_dataset(train_paths, HR_SIZE, UP_RATIO, BATCH_SIZE, training=True)
val_ds = build_dataset(val_paths, HR_SIZE, UP_RATIO, BATCH_SIZE, training=False)

val_ds = val_ds.cache()

show_cb = ShowExampleCallback(
    dataset=train_ds,
    every_n_epochs=5,
    up_ratio=UP_RATIO,
    hr_size=HR_SIZE,
    num_examples=1
)

early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=len(train_paths) // BATCH_SIZE,
    validation_steps=len(val_paths) // BATCH_SIZE,
    callbacks=[show_cb, early_stop]
)

In [ ]:
# Mostrar tantos ejemplos como quieras
for _ in range(5):
    show_example(model, train_ds, up_ratio=UP_RATIO)

In [ ]:
model.save(filepath="./ESPCN_x4.keras")